# DeepSets Extrapolation Training

This notebook trains DeepSets models on mixed-distance datasets (d=3, d=5, d=7) to study extrapolation to larger distances (d=9, d=11).

**Workflow:**
1. Load best hyperparameters from tuning results
2. Configure split experiments (different ratios of d=3/d=5/d=7 data)
3. Generate/load combined training datasets
4. Train models for each split configuration
5. Save trained models for evaluation in testing.ipynb

**Split Experiments (15 total):**

The experiments are organized into 4 groups to systematically study the effect of d=3 proportion and d5:d7 ratio:

**Control:**
- `equal_333333`: Baseline with equal distribution (33% d=3, 33% d=5, 34% d=7)

**Experiment A: Vary d=3 amount (d5:d7 = 1:1)**
- `a1_d3_00`: 0% d=3, 50% d=5, 50% d=7
- `a2_d3_10`: 10% d=3, 45% d=5, 45% d=7
- `a3_d3_20`: 20% d=3, 40% d=5, 40% d=7
- `a4_d3_40`: 40% d=3, 30% d=5, 30% d=7
- `a5_d3_50`: 50% d=3, 25% d=5, 25% d=7

**Experiment B: Vary d5:d7 ratio (no d=3)**
- `b1_d5heavy`: 0% d=3, 80% d=5, 20% d=7
- `b2_d5more`: 0% d=3, 65% d=5, 35% d=7
- `b3_balanced`: 0% d=3, 50% d=5, 50% d=7
- `b4_d7more`: 0% d=3, 35% d=5, 65% d=7
- `b5_d7heavy`: 0% d=3, 20% d=5, 80% d=7

**Experiment C: Extreme/boundary cases**
- `c1_only_d3`: 100% d=3, 0% d=5, 0% d=7
- `c2_only_d5`: 0% d=3, 100% d=5, 0% d=7
- `c3_only_d7`: 0% d=3, 0% d=5, 100% d=7
- `c4_no_d7`: 50% d=3, 50% d=5, 0% d=7

**Worker System:**
Experiments are divided into 3 workers (5 experiments each):
- **Worker 1**: equal_333333, a1_d3_00, a2_d3_10, a3_d3_20, a4_d3_40
- **Worker 2**: a5_d3_50, b1_d5heavy, b2_d5more, b3_balanced, b4_d7more
- **Worker 3**: b5_d7heavy, c1_only_d3, c2_only_d5, c3_only_d7, c4_no_d7

## Imports

In [ ]:
!pip install stim pymatching numpy matplotlib torch tqdm

In [ ]:
import sys
import json
import random
import time
from pathlib import Path
from datetime import datetime

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/deepsets/extrapolation -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

# Import from benchmark_models.py
from benchmark_models import DeepSets, SurfaceCodeSampler

# Set up paths
EXTRAPOLATION_DIR = BASE_PATH / "deepsets" / "extrapolation"
EXTRAPOLATION_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = EXTRAPOLATION_DIR / "results" / "revised_training"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = EXTRAPOLATION_DIR / "models" / "revised_training"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = EXTRAPOLATION_DIR / "plots" / "revised_training"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  EXTRAPOLATION_DIR: {EXTRAPOLATION_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  MODELS_DIR: {MODELS_DIR}")

## Configuration

In [ ]:
# =============================================================================
# BEST HYPERPARAMETERS (loaded from tuning results)
# =============================================================================

# Load best hyperparameters from tuning results
BEST_CONFIG_PATH = BASE_PATH / "deepsets" / "tuning" / "best_model_config.json"

if BEST_CONFIG_PATH.exists():
    with open(BEST_CONFIG_PATH, 'r') as f:
        best_config = json.load(f)
    
    # Extract hyperparameters from the config file
    BEST_HYPERPARAMS = {
        'phi_hidden': tuple(best_config['model_params']['phi_hidden']),
        'rho_hidden': tuple(best_config['model_params']['rho_hidden']),
        'pool': best_config['model_params']['pool'],
        'dropout': best_config['model_params'].get('dropout', 0.0),
        'learning_rate': best_config['training_params']['learning_rate'],
    }
    
    print(f"Loaded best hyperparameters from: {BEST_CONFIG_PATH}")
    if 'config_id' in best_config:
        print(f"Config ID: {best_config['config_id']}")
    if 'performance' in best_config:
        print(f"Tuning performance: val_acc={best_config['performance']['val_accuracy']*100:.2f}%, test_acc={best_config['performance']['test_accuracy']*100:.2f}%")
else:
    print("No best config found. Using defaults.")
    BEST_HYPERPARAMS = {
        'phi_hidden': (128, 128),
        'rho_hidden': (256, 128),
        'pool': 'sum',
        'dropout': 0.0,
        'learning_rate': 5e-4,
    }

print(f"\nHyperparameters:")
for k, v in BEST_HYPERPARAMS.items():
    print(f"  {k}: {v}")

In [ ]:
# =============================================================================
# TRAINING CONFIGURATION
# =============================================================================

# Total samples defining dataset scale (10^6)
TOTAL_SAMPLES = 1_000_000

# Training parameters
EPOCHS = 50
BATCH_SIZE = 256

# Refresh/eval cadence
REFRESH_EVERY = 10          # regenerate TRAINING data every N epochs
VALIDATE_EVERY = 10         # run validation every N epochs (at refresh boundaries)

# Split ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Training samples per refresh block
TRAIN_SAMPLES_PER_BLOCK = int(TOTAL_SAMPLES * TRAIN_RATIO)

# Eval set size built from cached baseline datasets (we only need val+test)
EVAL_TOTAL = int(TOTAL_SAMPLES * (VAL_RATIO + TEST_RATIO))

# Error rate distribution for generated TRAINING data
P_VALUES = [0.001, 0.003, 0.005, 0.007]
P_WEIGHTS = [0.25, 0.25, 0.25, 0.25]

# Random seed for reproducibility
SEED = 42

print(f"Training Configuration:")
print(f"  Total samples: {TOTAL_SAMPLES:,}")
print(f"  Training samples per refresh block: {TRAIN_SAMPLES_PER_BLOCK:,}")
print(f"  Eval total (val+test) samples: {EVAL_TOTAL:,}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Refresh every: {REFRESH_EVERY} epochs")
print(f"  Validate every: {VALIDATE_EVERY} epochs")
print(f"  P values: {P_VALUES}")
print(f"  P weights: {P_WEIGHTS}")

In [ ]:
# =============================================================================
# SPLIT EXPERIMENT CONFIGURATIONS
# =============================================================================

SPLIT_EXPERIMENTS = {
    # Control
    'equal_333333': {'d3': 0.33, 'd5': 0.33, 'd7': 0.34},
    
    # Experiment A: Vary d=3 amount (d5:d7 = 1:1)
    'a1_d3_00': {'d3': 0.00, 'd5': 0.50, 'd7': 0.50},
    'a2_d3_10': {'d3': 0.10, 'd5': 0.45, 'd7': 0.45},
    'a3_d3_20': {'d3': 0.20, 'd5': 0.40, 'd7': 0.40},
    'a4_d3_40': {'d3': 0.40, 'd5': 0.30, 'd7': 0.30},
    'a5_d3_50': {'d3': 0.50, 'd5': 0.25, 'd7': 0.25},
    
    # Experiment B: Vary d5:d7 ratio (no d=3)
    'b1_d5heavy': {'d3': 0.00, 'd5': 0.80, 'd7': 0.20},
    'b2_d5more':  {'d3': 0.00, 'd5': 0.65, 'd7': 0.35},
    'b3_balanced': {'d3': 0.00, 'd5': 0.50, 'd7': 0.50},
    'b4_d7more':  {'d3': 0.00, 'd5': 0.35, 'd7': 0.65},
    'b5_d7heavy': {'d3': 0.00, 'd5': 0.20, 'd7': 0.80},
    
    # Experiment C: Extreme/boundary cases
    'c1_only_d3': {'d3': 1.00, 'd5': 0.00, 'd7': 0.00},
    'c2_only_d5': {'d3': 0.00, 'd5': 1.00, 'd7': 0.00},
    'c3_only_d7': {'d3': 0.00, 'd5': 0.00, 'd7': 1.00},
    'c4_no_d7':   {'d3': 0.50, 'd5': 0.50, 'd7': 0.00},
}

print("Split Experiments:")
print(f"{'Name':<25} {'d3':>6} {'d5':>6} {'d7':>6}")
print("-" * 50)
for name, config in SPLIT_EXPERIMENTS.items():
    print(f"{name:<25} {config['d3']*100:>5.0f}% {config['d5']*100:>5.0f}% {config['d7']*100:>5.0f}%")

In [ ]:
# =============================================================================
# SELECT WHICH EXPERIMENT TO RUN
# =============================================================================

# Worker assignments (each worker trains 5 experiments)
WORKER_EXPERIMENTS = {
    1: ['equal_333333', 'a1_d3_00', 'a2_d3_10', 'a3_d3_20', 'a4_d3_40'],
    2: ['a5_d3_50', 'b1_d5heavy', 'b2_d5more', 'b3_balanced', 'b4_d7more'],
    3: ['b5_d7heavy', 'c1_only_d3', 'c2_only_d5', 'c3_only_d7', 'c4_no_d7'],
}

# Select which worker to run (1, 2, or 3)
WORKER = 1  # Change this to 1, 2, or 3

if WORKER not in WORKER_EXPERIMENTS:
    raise ValueError(f"WORKER must be 1, 2, or 3, got {WORKER}")

experiments_to_run = WORKER_EXPERIMENTS[WORKER]
print(f"Worker {WORKER} will train {len(experiments_to_run)} experiments:")
for exp in experiments_to_run:
    print(f"  - {exp}")

## Helper Functions

In [ ]:
def _mix_seed(base: int, scope: str, block_idx: int) -> int:
    """Deterministic seed mixer (uint32-ish)."""
    h = base
    for ch in str(scope):
        h = (h * 1000003 + ord(ch)) % (2**32 - 1)
    h = (h * 1009 + block_idx * 9176) % (2**32 - 1)
    return int(h)


def load_cached_eval_data_for_split(split_name: str, split_config: dict, total_eval: int) -> tuple:
    """Build fixed (val_data, test_data) from cached baseline datasets.

    We only materialize val+test data (default 200k), not the full 1e6.
    Returns dict format: {'detections': tensor, 'labels': tensor, 'distances': tensor}
    """
    NN_DATASETS_DIR = BASE_PATH / "nn_datasets"
    
    # Determine counts by proportion (last distance gets remainder)
    items = [('d3', 3), ('d5', 5), ('d7', 7)]
    counts = []
    allocated = 0
    for key, d in items:
        prop = float(split_config.get(key, 0.0))
        n = int(total_eval * prop)
        counts.append((d, n))
        allocated += n
    # Add remainder to the last non-zero distance (or to d7 by default)
    remainder = total_eval - allocated
    if remainder != 0:
        for i in range(len(counts)-1, -1, -1):
            d_i, n_i = counts[i]
            if n_i > 0 or d_i == 7:
                counts[i] = (d_i, n_i + remainder)
                break

    all_detections = []
    all_labels = []
    all_distances = []
    
    seed_i = _mix_seed(SEED, f"eval_{split_name}", 0)
    random.seed(seed_i)

    for d, n in counts:
        if n <= 0:
            continue
        cache_path = NN_DATASETS_DIR / f"d{d}_baseline_array.pt"
        if not cache_path.exists():
            raise FileNotFoundError(f"Cached dataset not found: {cache_path}")
        
        data = torch.load(cache_path, weights_only=False)
        total_available = len(data['labels'])
        if total_available < n:
            raise ValueError(f"Cached dataset '{cache_path}' has {total_available:,} samples, need {n:,} for eval")
        
        # Randomly sample n indices
        indices = torch.randperm(total_available)[:n]
        all_detections.append(data['detections'][indices].float())
        all_labels.append(data['labels'][indices].float())
        all_distances.append(torch.full((n,), d, dtype=torch.long))
        print(f"  Loaded {n:,} eval samples from d={d}")

    # Combine and shuffle
    combined_det = torch.cat(all_detections, dim=0)
    combined_lab = torch.cat(all_labels, dim=0)
    combined_dist = torch.cat(all_distances, dim=0)
    
    # Shuffle together
    perm = torch.randperm(len(combined_lab))
    combined_det = combined_det[perm]
    combined_lab = combined_lab[perm]
    combined_dist = combined_dist[perm]

    # Split eval into val/test equally (since VAL_RATIO == TEST_RATIO)
    half = len(combined_lab) // 2
    val_data = {
        'detections': combined_det[:half],
        'labels': combined_lab[:half],
        'distances': combined_dist[:half]
    }
    test_data = {
        'detections': combined_det[half:],
        'labels': combined_lab[half:],
        'distances': combined_dist[half:]
    }

    print(f"Eval data built from cache for {split_name}: {len(val_data['labels']):,} val, {len(test_data['labels']):,} test")
    return val_data, test_data


def generate_fresh_training_data_for_split(split_name: str, split_config: dict, total_train: int, block_idx: int) -> list:
    """Generate a fresh TRAINING dataset for this split (no disk cache).
    
    Returns list of tuples: [(detections, labels, distance), ...] for each distance group.
    """
    print(f"\n[train refresh] {split_name}: generating training data for block {block_idx} (total={total_train:,})")

    seed_i = _mix_seed(SEED, f"train_{split_name}", block_idx)
    random.seed(seed_i)
    np.random.seed(seed_i % (2**32 - 1))
    torch.manual_seed(seed_i)

    items = [('d3', 3), ('d5', 5), ('d7', 7)]
    counts = []
    allocated = 0
    for key, d in items:
        prop = float(split_config.get(key, 0.0))
        n = int(total_train * prop)
        counts.append((d, n))
        allocated += n
    remainder = total_train - allocated
    if remainder != 0:
        # put remainder into the largest-distance bucket by default (d7)
        for i in range(len(counts)-1, -1, -1):
            d_i, n_i = counts[i]
            if n_i > 0 or d_i == 7:
                counts[i] = (d_i, n_i + remainder)
                break

    train_groups = []
    sampler = SurfaceCodeSampler(p=0.001, device=device)
    
    for d, n in counts:
        if n <= 0:
            continue

        print(f"  Generating {n:,} samples for d={d}...")
        det, lab = sampler.sample(
            d=d,
            num_samples=n,
            p_values=P_VALUES,
            p_weights=P_WEIGHTS
        )
        train_groups.append((det.float(), lab.float(), d))

    total_generated = sum(len(g[1]) for g in train_groups)
    print(f"Training data ready: {total_generated:,} samples")
    return train_groups

In [ ]:
def evaluate_model(model, eval_data: dict, device, batch_size=256):
    """
    Evaluate model accuracy on evaluation data.
    
    Args:
        model: DeepSets model wrapper
        eval_data: Dict with 'detections', 'labels', 'distances' tensors
        device: Torch device
        batch_size: Batch size for evaluation
        
    Returns:
        Accuracy as a float
    """
    model.model.eval()
    
    detections = eval_data['detections']
    labels = eval_data['labels']
    distances = eval_data['distances']
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for start in range(0, len(labels), batch_size):
            end = min(start + batch_size, len(labels))
            batch_det = detections[start:end].to(device)
            batch_lab = labels[start:end].to(device).unsqueeze(-1)
            batch_dist = distances[start:end]
            
            # Get unique distances in this batch
            unique_dists = batch_dist.unique()
            
            # Process each distance group separately
            batch_correct = 0
            batch_total = 0
            
            for d in unique_dists:
                mask = batch_dist == d
                d_det = batch_det[mask]
                d_lab = batch_lab[mask]
                
                coords, counts = model._syndromes_to_coords(d_det, d.item())
                pred = model.model(coords, counts)
                
                batch_correct += ((pred > 0.5).float() == d_lab).sum().item()
                batch_total += d_lab.size(0)
            
            correct += batch_correct
            total += batch_total
    
    return correct / total if total > 0 else 0.0

In [ ]:
def save_model(model, split_name: str, metrics: dict):
    """
    Save trained model with metadata.
    
    Args:
        model: DeepSets model wrapper
        split_name: Name of the split experiment
        metrics: Dict with training metrics
    """
    model_path = MODELS_DIR / f"{split_name}.pt"
    
    save_dict = {
        'state_dict': model.model.state_dict(),
        'config': model._config,
        'split_name': split_name,
        'split_config': SPLIT_EXPERIMENTS[split_name],
        'hyperparams': BEST_HYPERPARAMS,
        'metrics': metrics,
        'timestamp': datetime.now().isoformat(),
    }
    
    torch.save(save_dict, model_path)
    print(f"Model saved to: {model_path}")
    return model_path


def save_training_results(split_name: str, epoch_metrics: dict, final_metrics: dict = None):
    """Save training results to JSON with full epoch-by-epoch metrics."""
    results_path = RESULTS_DIR / f"{split_name}_training.json"

    results = {
        'split_name': split_name,
        'split_config': SPLIT_EXPERIMENTS[split_name],
        'hyperparams': BEST_HYPERPARAMS,
        'training_config': {
            'total_samples': TOTAL_SAMPLES,
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'refresh_every': REFRESH_EVERY,
            'validate_every': VALIDATE_EVERY,
            'train_samples_per_block': TRAIN_SAMPLES_PER_BLOCK,
            'eval_total': EVAL_TOTAL,
            'p_values': P_VALUES,
            'p_weights': P_WEIGHTS,
        },
        'epoch_metrics': {
            'epochs': epoch_metrics.get('epoch', []),
            'train_loss': epoch_metrics.get('train_loss', []),
            'train_accuracy': epoch_metrics.get('train_accuracy', []),
            'val_accuracy': epoch_metrics.get('val_accuracy', []),
            'epoch_time_seconds': epoch_metrics.get('epoch_time_seconds', []),
            'learning_rate': epoch_metrics.get('learning_rate', []),
            'gpu_memory_mb': epoch_metrics.get('gpu_memory_mb', []),
            'train_block_idx': epoch_metrics.get('train_block_idx', []),
            'train_samples': epoch_metrics.get('train_samples', []),
        },
        'final_metrics': final_metrics if final_metrics is not None else {},
        'timestamp': datetime.now().isoformat(),
    }

    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"Results saved to: {results_path}")

In [ ]:
# =============================================================================
# CHECKPOINTING (per split)
# =============================================================================

def _checkpoint_path(split_name: str) -> Path:
    return MODELS_DIR / f"{split_name}_checkpoint.pt"


def load_checkpoint(split_name: str) -> dict:
    path = _checkpoint_path(split_name)
    if not path.exists():
        return None
    print(f"Loading checkpoint: {path}")
    return torch.load(path, map_location=device, weights_only=False)


def save_checkpoint(split_name: str, checkpoint: dict):
    path = _checkpoint_path(split_name)
    torch.save(checkpoint, path)

In [ ]:
# =============================================================================
# TRAINING LOOP (refresh every N epochs)
# =============================================================================

def train_with_refresh(
    split_name: str,
    split_config: dict,
    model: DeepSets,
    val_data: dict,
    epochs: int,
    batch_size: int,
    lr: float,
    refresh_every: int,
    validate_every: int,
    start_epoch: int = 0,
    initial_metrics: dict = None,
    optimizer_state_dict: dict = None,
    verbose: bool = True,
):
    import gc

    optimizer = torch.optim.Adam(model.model.parameters(), lr=lr)
    loss_fn = torch.nn.BCELoss()

    if optimizer_state_dict is not None:
        optimizer.load_state_dict(optimizer_state_dict)
        if verbose:
            print("Restored optimizer state from checkpoint")

    if initial_metrics is not None:
        epoch_metrics = initial_metrics.copy()
    else:
        epoch_metrics = {
            'epoch': [],
            'train_loss': [],
            'train_accuracy': [],
            'val_accuracy': [],
            'epoch_time_seconds': [],
            'train_block_idx': [],
            'learning_rate': [],
            'gpu_memory_mb': [],
            'train_samples': [],
        }

    total_start_time = time.time()

    current_block_idx = start_epoch // refresh_every
    train_groups = generate_fresh_training_data_for_split(split_name, split_config, TRAIN_SAMPLES_PER_BLOCK, current_block_idx)

    for epoch in range(start_epoch, epochs):
        epoch_start = time.time()

        desired_block_idx = epoch // refresh_every
        if desired_block_idx != current_block_idx:
            current_block_idx = desired_block_idx
            del train_groups
            gc.collect()
            train_groups = generate_fresh_training_data_for_split(split_name, split_config, TRAIN_SAMPLES_PER_BLOCK, current_block_idx)

        model.model.train()
        
        # Create batches from all distance groups
        all_batches = []
        for det, lab, d in train_groups:
            n = len(lab)
            start_idx = 0
            while start_idx < n:
                end_idx = min(start_idx + batch_size, n)
                all_batches.append((det[start_idx:end_idx], lab[start_idx:end_idx], d))
                start_idx = end_idx
        
        random.shuffle(all_batches)
        
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0

        pbar = tqdm(all_batches, desc=f"Epoch {epoch+1}/{epochs}", disable=not verbose, leave=False)
        for batch_det, batch_lab, d in pbar:
            batch_det = batch_det.to(device)
            batch_lab = batch_lab.to(device).unsqueeze(-1)
            
            coords, counts = model._syndromes_to_coords(batch_det, d)
            pred = model.model(coords, counts)

            loss = loss_fn(pred, batch_lab)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            epoch_correct += ((pred > 0.5).float() == batch_lab).sum().item()
            epoch_total += batch_lab.size(0)

            current_acc = 100.0 * epoch_correct / epoch_total
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{current_acc:.1f}%'})

        pbar.close()

        avg_train_loss = epoch_loss / max(1, len(all_batches))
        train_acc = epoch_correct / max(1, epoch_total)

        do_validate = ((epoch + 1) % validate_every == 0) or (epoch + 1 == epochs)
        if do_validate:
            val_acc = evaluate_model(model, val_data, device, batch_size=batch_size)
        else:
            val_acc = float('nan')

        epoch_time = time.time() - epoch_start

        # GPU memory tracking
        if torch.cuda.is_available():
            gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
            torch.cuda.reset_peak_memory_stats()
        else:
            gpu_memory_mb = 0.0

        # Learning rate from optimizer
        current_lr = optimizer.param_groups[0]['lr']

        # Training samples
        train_samples_count = sum(len(g[1]) for g in train_groups)

        epoch_metrics['epoch'].append(epoch + 1)
        epoch_metrics['train_loss'].append(avg_train_loss)
        epoch_metrics['train_accuracy'].append(train_acc)
        epoch_metrics['val_accuracy'].append(val_acc)
        epoch_metrics['epoch_time_seconds'].append(epoch_time)
        epoch_metrics['train_block_idx'].append(current_block_idx)
        epoch_metrics['learning_rate'].append(current_lr)
        epoch_metrics['gpu_memory_mb'].append(gpu_memory_mb)
        epoch_metrics['train_samples'].append(train_samples_count)

        if verbose:
            val_str = f"{val_acc*100:.2f}%" if not np.isnan(val_acc) else "--"
            print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_train_loss:.4f}, Train Acc: {train_acc*100:.2f}%, Val Acc: {val_str}")

        # checkpoint every epoch
        save_checkpoint(split_name, {
            'state_dict': model.model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'current_epoch': epoch + 1,
            'epoch_metrics': epoch_metrics,
            'refresh_every': refresh_every,
            'validate_every': validate_every,
            'current_block_idx': current_block_idx,
            'timestamp': datetime.now().isoformat(),
        })

    total_time = time.time() - total_start_time
    epoch_metrics['total_training_time_seconds'] = total_time
    epoch_metrics['_optimizer_state_dict'] = optimizer.state_dict()

    return epoch_metrics

## Training Pipeline

In [ ]:
def run_split_experiment(split_name: str, split_config: dict):
    """Run one extrapolation split with training refresh + cached eval."""
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {split_name}")
    print(f"{'='*70}")
    print(f"Split: d3={split_config['d3']*100:.0f}%, d5={split_config['d5']*100:.0f}%, d7={split_config['d7']*100:.0f}%")
    print(f"{'='*70}")

    start_time = time.time()

    # 1) Cached EVAL (val/test) built from baseline caches
    print("\n[1/4] Building cached eval datasets (val/test)...")
    val_data, test_data = load_cached_eval_data_for_split(split_name, split_config, total_eval=EVAL_TOTAL)

    # 2) Initialize model
    print("\n[2/4] Initializing model...")
    model = DeepSets(
        nickname=f"extrapolation_{split_name}",
        phi_hidden=BEST_HYPERPARAMS['phi_hidden'],
        rho_hidden=BEST_HYPERPARAMS['rho_hidden'],
        pool=BEST_HYPERPARAMS['pool'],
        dropout=BEST_HYPERPARAMS['dropout'],
        device=device,
        base_path=BASE_PATH,
        seed=SEED
    )

    # 3) Resume if checkpoint exists
    checkpoint = load_checkpoint(split_name)
    start_epoch = 0
    initial_metrics = None
    optimizer_state_dict = None

    if checkpoint is not None:
        start_epoch = int(checkpoint.get('current_epoch', 0))
        initial_metrics = checkpoint.get('epoch_metrics', None)
        optimizer_state_dict = checkpoint.get('optimizer_state_dict', None)
        if 'state_dict' in checkpoint:
            model.model.load_state_dict(checkpoint['state_dict'])
        if start_epoch >= EPOCHS:
            print(f"Checkpoint indicates training complete at epoch {start_epoch}. Skipping training.")

    # 4) Train with refresh
    print("\n[3/4] Training model (refresh training data every 10 epochs)...")

    if start_epoch < EPOCHS:
        epoch_metrics = train_with_refresh(
            split_name=split_name,
            split_config=split_config,
            model=model,
            val_data=val_data,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            lr=BEST_HYPERPARAMS['learning_rate'],
            refresh_every=REFRESH_EVERY,
            validate_every=VALIDATE_EVERY,
            start_epoch=start_epoch,
            initial_metrics=initial_metrics,
            optimizer_state_dict=optimizer_state_dict,
            verbose=True,
        )
    else:
        epoch_metrics = initial_metrics
        if epoch_metrics is None:
            epoch_metrics = {
                'epoch': [],
                'train_loss': [],
                'train_accuracy': [],
                'val_accuracy': [],
                'epoch_time_seconds': [],
                'train_block_idx': [],
                'learning_rate': [],
                'gpu_memory_mb': [],
                'train_samples': [],
            }

    # Evaluate
    print("\n[4/4] Evaluating model...")
    last_block_idx = (EPOCHS - 1) // REFRESH_EVERY
    train_eval_groups = generate_fresh_training_data_for_split(split_name, split_config, total_train=10_000, block_idx=last_block_idx)
    
    # Convert train_eval_groups to dict format for evaluate_model
    train_eval_det = torch.cat([g[0] for g in train_eval_groups], dim=0)
    train_eval_lab = torch.cat([g[1] for g in train_eval_groups], dim=0)
    train_eval_dist = torch.cat([torch.full((len(g[1]),), g[2], dtype=torch.long) for g in train_eval_groups], dim=0)
    train_eval_data = {'detections': train_eval_det, 'labels': train_eval_lab, 'distances': train_eval_dist}

    train_acc = evaluate_model(model, train_eval_data, device, batch_size=BATCH_SIZE)
    val_acc = evaluate_model(model, val_data, device, batch_size=BATCH_SIZE)
    test_acc = evaluate_model(model, test_data, device, batch_size=BATCH_SIZE)

    training_time = time.time() - start_time

    epoch_losses = epoch_metrics.get('train_loss', [])
    final_loss = epoch_losses[-1] if epoch_losses else None

    # Prepare final metrics for model save (backward compatibility)
    metrics = {
        'epoch_losses': epoch_losses,
        'final_loss': final_loss,
        'train_accuracy': train_acc,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'training_time_seconds': training_time,
    }

    # Prepare final_metrics for results JSON
    final_metrics = {
        'final_loss': final_loss,
        'train_accuracy': train_acc,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'total_training_time_seconds': training_time,
    }

    save_model(model, split_name, metrics)
    save_training_results(split_name, epoch_metrics, final_metrics)

    print(f"\n{'='*70}")
    print(f"EXPERIMENT COMPLETE: {split_name}")
    print(f"{'='*70}")
    print(f"  Training time: {training_time/60:.1f} minutes")
    if final_loss is not None:
        print(f"  Final loss: {final_loss:.4f}")
    print(f"  Train accuracy: {train_acc*100:.2f}%")
    print(f"  Val accuracy: {val_acc*100:.2f}%")
    print(f"  Test accuracy: {test_acc*100:.2f}%")
    print(f"{'='*70}")

    # Clean up memory
    del model, val_data, test_data, train_eval_data
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return metrics

## Run Training Experiments

In [ ]:
# Confirm hyperparameters loaded successfully
print(f"Using hyperparameters:")
for k, v in BEST_HYPERPARAMS.items():
    print(f"  {k}: {v}")

print(f"\n{'#'*70}")
print(f"STARTING EXTRAPOLATION TRAINING EXPERIMENTS")
print(f"{'#'*70}")
print(f"Experiments to run: {len(experiments_to_run)}")
for exp in experiments_to_run:
    print(f"  - {exp}")

In [ ]:
# Run all selected experiments
all_results = {}

for i, split_name in enumerate(experiments_to_run):
    print(f"\n\n{'#'*70}")
    print(f"EXPERIMENT {i+1}/{len(experiments_to_run)}: {split_name}")
    print(f"{'#'*70}")
    
    split_config = SPLIT_EXPERIMENTS[split_name]
    metrics = run_split_experiment(split_name, split_config)
    all_results[split_name] = metrics

print(f"\n\n{'#'*70}")
print(f"ALL EXPERIMENTS COMPLETE")
print(f"{'#'*70}")

## Results Summary

In [ ]:
# Create summary table
if all_results:
    summary_data = []
    for split_name, metrics in all_results.items():
        config = SPLIT_EXPERIMENTS[split_name]
        summary_data.append({
            'Split': split_name,
            'd3 %': config['d3'] * 100,
            'd5 %': config['d5'] * 100,
            'd7 %': config['d7'] * 100,
            'Train Acc': metrics['train_accuracy'] * 100,
            'Val Acc': metrics['val_accuracy'] * 100,
            'Test Acc': metrics['test_accuracy'] * 100,
            'Final Loss': metrics['final_loss'],
            'Time (min)': metrics['training_time_seconds'] / 60,
        })
    
    df_summary = pd.DataFrame(summary_data)
    print("\nTraining Results Summary:")
    print(df_summary.to_string(index=False))
    
    # Save summary
    summary_path = RESULTS_DIR / "training_summary.csv"
    df_summary.to_csv(summary_path, index=False)
    print(f"\nSummary saved to: {summary_path}")

In [ ]:
# Plot training curves
if all_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    ax1 = axes[0]
    for split_name, metrics in all_results.items():
        ax1.plot(metrics['epoch_losses'], label=split_name)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss by Split Configuration')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy comparison
    ax2 = axes[1]
    split_names = list(all_results.keys())
    val_accs = [all_results[s]['val_accuracy'] * 100 for s in split_names]
    test_accs = [all_results[s]['test_accuracy'] * 100 for s in split_names]
    
    x = np.arange(len(split_names))
    width = 0.35
    ax2.bar(x - width/2, val_accs, width, label='Validation')
    ax2.bar(x + width/2, test_accs, width, label='Test')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Accuracy by Split Configuration')
    ax2.set_xticks(x)
    ax2.set_xticklabels(split_names, rotation=45, ha='right')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    # Save plot
    plot_path = PLOTS_DIR / "training_results.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"Plot saved to: {plot_path}")
    
    plt.show()

## Next Steps

After training completes:
1. Review the training results summary above
2. Run `testing.ipynb` to evaluate extrapolation to d=9 and d=11
3. Compare how different split configurations affect extrapolation performance

In [ ]:
from google.colab import runtime
runtime.unassign()